In [2]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

# ================= CONFIGURATION =================
SOURCE_DIR = 'Images'
DEST_TRAIN_DIR = 'HR'
DEST_TEST_DIR = '../testsets/UCMerced_LandUse/HR'

# Split ratio (e.g., 0.2 means 20% of images from EACH class go to test)
TEST_SPLIT_RATIO = 0.2

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.tif', '.bmp'}
# =================================================

def split_and_extract(source, train_dest, test_dest, split_ratio):
    source_path = Path(source)
    train_path = Path(train_dest)
    test_path = Path(test_dest)

    # Create directories
    train_path.mkdir(parents=True, exist_ok=True)
    test_path.mkdir(parents=True, exist_ok=True)

    # 1. Identify all class subdirectories
    classes = [d for d in source_path.iterdir() if d.is_dir()]

    total_train = 0
    total_test = 0

    print(f"Found {len(classes)} classes. Starting split...")

    for class_dir in tqdm(classes, desc="Processing Classes"):
        # 2. Get all images in this specific class folder
        images = [f for f in class_dir.glob('*') if f.suffix.lower() in IMG_EXTENSIONS]

        # Shuffle to ensure random sampling
        random.shuffle(images)

        # 3. Calculate split index for this class
        num_test = int(len(images) * split_ratio)
        test_images = images[:num_test]
        train_images = images[num_test:]

        # 4. Helper function to copy files and avoid collisions
        def copy_files(file_list, target_folder):
            moved_count = 0
            for img_p in file_list:
                # Use class name prefix to ensure uniqueness across the whole dataset
                unique_name = f"{class_dir.name}_{img_p.name}"
                dest_file = target_folder / unique_name

                shutil.copy2(img_p, dest_file)
                moved_count += 1
            return moved_count

        total_test += copy_files(test_images, test_path)
        total_train += copy_files(train_images, train_path)

    print(f"\n--- Processing Complete ---")
    print(f"Train Images: {total_train}")
    print(f"Test Images:  {total_test}")
    print(f"Split Ratio:  {split_ratio * 100}% test / {(1-split_ratio)*100}% train per class")

if __name__ == "__main__":
    split_and_extract(SOURCE_DIR, DEST_TRAIN_DIR, DEST_TEST_DIR, TEST_SPLIT_RATIO)

Found 21 classes. Starting split...


Processing Classes: 100%|██████████| 21/21 [00:01<00:00, 10.88it/s]


--- Processing Complete ---
Train Images: 1680
Test Images:  420
Split Ratio:  20.0% test / 80.0% train per class
